In [ ]:
# Genel veriyi temizliyor.

import pandas as pd
import numpy as np

# 1. DOSYAYI OKU
filename = 'human_vital_signs_dataset_2024.csv'
try:
    df = pd.read_csv(filename, sep=';')
    print("Dosya başarıyla yüklendi.")
except Exception as e:
    df = pd.read_csv(filename)
    print("Düz okuma yapıldı.")

# 2. SÜTUN İSİMLERİNİ TEMİZLE
df.columns = df.columns.str.strip()

# 3. SAYI DÜZELTME FONKSİYONU
def sayı_duzelt(deger, tip="genel"):
    if pd.isna(deger): return deger
    s_deger = str(deger).replace(',', '').replace('.', '').replace(' ', '')
    if not s_deger.isdigit(): return 0.0

    try:
        if tip == "temp": return float(s_deger[:2] + '.' + s_deger[2:4])
        elif tip == "oxy": return float(s_deger[:2] + '.' + s_deger[2:4])
        elif tip == "height": return float(s_deger[0] + '.' + s_deger[1:3])
        elif tip == "weight": return float(s_deger[:2] + '.' + s_deger[2:4])
        else: return float(s_deger)
    except:
        return 0.0

# 4. SÜTUNLARI BUL VE SAYIYA ÇEVİR
try:
    # Sütunları yakala
    temp_col = [c for c in df.columns if 'Temp' in c][0]
    oxy_col = [c for c in df.columns if 'Oxy' in c or 'SpO2' in c][0]
    height_col = [c for c in df.columns if 'Height' in c][0]
    weight_col = [c for c in df.columns if 'Weight' in c][0]
    hr_col = [c for c in df.columns if 'Heart' in c or 'Pulse' in c][0]
    age_col = [c for c in df.columns if 'Age' in c][0]
    gen_col = [c for c in df.columns if 'Gender' in c][0]

    # Sayıları temizle
    df[temp_col] = df[temp_col].apply(lambda x: sayı_duzelt(x, "temp"))
    df[oxy_col] = df[oxy_col].apply(lambda x: sayı_duzelt(x, "oxy"))
    df[height_col] = df[height_col].apply(lambda x: sayı_duzelt(x, "height")).astype(float)
    df[weight_col] = df[weight_col].apply(lambda x: sayı_duzelt(x, "weight")).astype(float)

    # 5. BMI HESAPLAMA
    df['Derived_BMI'] = df[weight_col] / (df[height_col] ** 2)

    # 6. SÜTUNLARI FİLTRELE VE GEREKSİZLERİ SİL (Kritik Değişiklik)
    # Sadece modelin beklediği 8 temel sütunu tutuyoruz (HRV eğitim kodunda eklenecek)
    target_columns = {
        hr_col: 'Heart Rate',
        temp_col: 'Body Temperature',
        oxy_col: 'Oxygen Saturation',
        age_col: 'Age',
        gen_col: 'Gender',
        weight_col: 'Weight (kg)',
        height_col: 'Height (m)',
        'Derived_BMI': 'Derived_BMI'
    }

    # Sütunları yeniden adlandır ve sadece bunları seç
    df_final = df[list(target_columns.keys())].rename(columns=target_columns)

    print("--- TEMİZLİK BAŞARILI ---")
    print(f"Yeni Sütunlar: {df_final.columns.tolist()}")

    # Temiz dosyayı kaydet
    df_final.to_csv('temiz_veriseti.csv', index=False)

except Exception as e:
    print(f"Hata: {e}")

In [ ]:
# Temizlenen verileri, girdiğimiz değer aralıklarından 0 ve 1 diye risk_status ekliyor.

import pandas as pd

# 1. Dosyayı Yükle
df = pd.read_csv('temiz_veriseti.csv')

# 2. Gerekli Hesaplamaları Yap (Sadece BMI kaldı, HRV kaldırıldı)
# Tüm değerleri 2 basamağa yuvarlıyoruz
df['Derived_BMI'] = (df['Weight (kg)'] / (df['Height (m)'] ** 2)).round(2)

# 3. Etiketleme Fonksiyonu
def binary_risk_label(row):
    # BMI hesapla
    current_bmi = row['Weight (kg)'] / (row['Height (m)'] ** 2)

    # Senin belirttiğin sınır değerleri
    is_oxygen_bad = row['Oxygen Saturation'] < 95
    is_heart_rate_bad = row['Heart Rate'] > 100 or row['Heart Rate'] < 60
    is_temp_bad = row['Body Temperature'] > 37.3 or row['Body Temperature'] < 35.5
    is_age_risky = row['Age'] >= 77
    is_bmi_bad = current_bmi > 30 or current_bmi < 18.5

    # Eğer bu 5 şarttan biri bile varsa RISK (1), yoksa (0)
    if is_oxygen_bad or is_heart_rate_bad or is_temp_bad or is_age_risky or is_bmi_bad:
        return 1
    return 0

# 4. Yeni Sütunu Uygula
df['Risk_Status'] = df.apply(binary_risk_label, axis=1)

# Cinsiyeti sayısal yap (Male:1, Female:0)
if df['Gender'].dtype == 'object':
    df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0}).fillna(0)

# 5. Tüm sayısal sütunları 2 basamağa yuvarla (Temiz bir görünüm için)
numeric_cols = df.select_dtypes(include=['float64']).columns
df[numeric_cols] = df[numeric_cols].round(2)

# 6. Yeni CSV Olarak Kaydet
df.to_csv('riskli_temiz_veriseti.csv', index=False)

print("İşlem tamamlandı! tüm sayılar 2 basamağa yuvarlandı.")
print(f"Toplam Satır: {len(df)}")
print(f"Riskli (1) Sayısı: {df['Risk_Status'].sum()}")
print(f"Normal (0) Sayısı: {len(df) - df['Risk_Status'].sum()}")

In [ ]:
# riskli_temiz_veriseti dosyasındaki 200k veriye 100k'lık veri daha ekliyor ve toplam 300k veri oluyor.
# Modelin daha iyi eğitilmesi için bunu yaptım.
# Verileri 00.00 bu tarz yazıp düzenliyor.

import pandas as pd
import numpy as np
import os

def append_medical_data_no_hrv(file_name='riskli_temiz_veriseti.csv', add_total=100000, normal_ratio=0.55):
    # 1. Mevcut veriyi oku
    if os.path.exists(file_name):
        df_existing = pd.read_csv(file_name)
        # Eğer mevcut dosyada HRV varsa, onu sütunlardan çıkarıyoruz
        if 'Derived_HRV' in df_existing.columns:
            df_existing = df_existing.drop(columns=['Derived_HRV'])
        print(f"📂 Mevcut dosya bulundu: {len(df_existing)} satır (HRV sütunu temizlendi).")
    else:
        df_existing = pd.DataFrame()
        print("⚠️ Dosya bulunamadı, yeni bir dosya oluşturulacak.")

    # 2. Yeni eklenecek veri sayılarını belirle
    target_normal = int(add_total * normal_ratio)
    target_risk = add_total - target_normal
    new_data = []
    counts = {0: 0, 1: 0}

    print(f"🔄 {add_total} yeni veri üretiliyor (8 Parametre, HRV Hariç)...")

    while sum(counts.values()) < add_total:
        # Rastgele değerler üret ve 2 basamağa yuvarla
        hr = round(np.random.uniform(50, 145), 2)
        temp = round(np.random.uniform(34.5, 40.5), 2)
        spo2 = round(np.random.uniform(80, 100), 2)
        age = np.random.randint(18, 95)
        gender = np.random.choice([0, 1])
        weight = round(np.random.uniform(45, 125), 2)
        height = round(np.random.uniform(1.50, 2.05), 2)

        # Sadece BMI hesapla (HRV kaldırıldı)
        bmi = round(weight / (height ** 2), 2)

        # Risk Mantığı (Senin 5 ana kuralın)
        is_risk = (spo2 < 95 or hr > 100 or hr < 60 or
                   temp > 37.3 or temp < 35.5 or
                   age >= 77 or bmi > 30 or bmi < 18.5)

        label = 1 if is_risk else 0

        if label == 0 and counts[0] < target_normal:
            new_data.append([hr, temp, spo2, age, gender, weight, height, bmi, label])
            counts[0] += 1
        elif label == 1 and counts[1] < target_risk:
            new_data.append([hr, temp, spo2, age, gender, weight, height, bmi, label])
            counts[1] += 1

    # 3. Yeni veriyi DataFrame yap (8 giriş özelliği + 1 etiket)
    df_new = pd.DataFrame(new_data, columns=[
        'Heart Rate', 'Body Temperature', 'Oxygen Saturation', 'Age',
        'Gender', 'Weight (kg)', 'Height (m)', 'Derived_BMI', 'Risk_Status'
    ])

    # 4. Mevcut veri ile birleştir
    df_final = pd.concat([df_existing, df_new], ignore_index=True)

    # 5. Tüm sayıları 2 basamağa yuvarla ve karıştır
    df_final = df_final.round(2)
    df_final = df_final.sample(frac=1).reset_index(drop=True)
    df_final.to_csv(file_name, index=False)

    print(f"✅ İŞLEM TAMAMLANDI!")
    print(f"📈 Toplam Satır Sayısı: {len(df_final)}")
    print(f"📊 Mevcut Sütunlar: {list(df_final.columns)}")
    print(f"📊 Riskli (1): {df_final['Risk_Status'].sum()}")
    print(f"📊 Normal (0): {len(df_final) - df_final['Risk_Status'].sum()}")

# Çalıştır
append_medical_data_no_hrv()

In [ ]:
# Modeli eğittim kod.
# Model artık böyle eğitiliyor Huni Modeli: 64-32-16. Eskiden 32-16-8'di galiba.
# Birde MEAN (Ortalamalar) ve SCALE (Standart Sapmalar) değerlerini bastıran bir dosya veiriyor.

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

df = pd.read_csv('riskli_temiz_veriseti.csv')

features = ['Heart Rate', 'Body Temperature', 'Oxygen Saturation', 'Age',
            'Gender', 'Weight (kg)', 'Height (m)', 'Derived_BMI']

X = df[features]
y = df['Risk_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("="*50)
print(f"TOPLAM VERİ SAYISI      : {len(df)}")
print(f"EĞİTİM SETİ (Train)     : {len(X_train)}")
print(f"TEST SETİ (Test)        : {len(X_test)}")
print(f"RİSKLİ ÖRNEK SAYISI (1) : {y.sum()}")
print(f"NORMAL ÖRNEK SAYISI (0) : {len(y) - y.sum()}")
print("="*50 + "\n")

model = tf.keras.Sequential([
    layers.Input(shape=(len(features),)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("🚀 Eğitim Başlıyor...")
model.fit(X_train_scaled, y_train,
          epochs=30,
          batch_size=64,
          verbose=1,
          validation_data=(X_test_scaled, y_test))

model.save('medical_binary_model.h5')
joblib.dump(scaler, 'scaler.pkl')
print("\n[TAMAM] Model ve Scaler kaydedildi!")

# ============================================
# YENİ: DETAYLI METRİKLER (eğitimi bozmaz)
# ============================================
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

print("\n" + "="*50)
print("📊 DETAYLI SINIFLANDIRMA RAPORU")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['NORMAL', 'RİSKLİ']))

cm = confusion_matrix(y_test, y_pred)
print("CONFUSION MATRIX:")
print(f"                 Tahmin NORMAL  Tahmin RİSKLİ")
print(f"Gerçek NORMAL  :     {cm[0][0]:<10}     {cm[0][1]}")
print(f"Gerçek RİSKLİ :     {cm[1][0]:<10}     {cm[1][1]}")

print("\n" + "="*50)

train_loss, train_acc = model.evaluate(X_train_scaled, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Eğitim Accuracy : %{train_acc*100:.2f}")
print(f"Test Accuracy   : %{test_acc*100:.2f}")
print(f"Fark            : %{abs(train_acc-test_acc)*100:.2f}")

print("\n--- GÜNCEL SCALER SABİTLERİ ---")
print("MEAN:", list(scaler.mean_))
print("SCALE:", list(scaler.scale_))


In [ ]:
# Eğittiğimiz modeli 8 parametre ile deniyoruz (HRV sadece ekranda görünecek).
# Toplam 100 örnek ile deniyoruz.

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model

# 1. Modeli Yükle (Artık 8 parametre bekleyen model)
model = load_model('medical_binary_model.h5')

# 2. KRİTİK NOT: 8 PARAMETRELİ EĞİTİM SONUNDA ALDIĞIN GÜNCEL SABİTLERİ BURAYA YAZMALISIN!
# Aşağıdakiler örnek 8'li değerlerdir, kendi eğitim çıktındaki 8'li listeyi buraya yapıştır.
means = np.array([82.334, 36.793, 96.362, 52.645, 0.499, 77.354, 1.761, 25.364]) # 8 Parametre
scales = np.array([16.357, 0.854, 3.749, 20.574, 0.500, 17.034, 0.151, 6.620])  # 8 Parametre

# 3. 50 Test Senaryosu (Ham Veriler)
raw_scenarios = [
    [72, 36.6, 98, 25, 1, 75, 1.80], [105, 36.8, 97, 30, 0, 60, 1.65], [65, 36.5, 92, 45, 1, 85, 1.75], [80, 38.2, 96, 12, 0, 40, 1.50], [70, 36.7, 96, 78, 1, 70, 1.70],
    [55, 36.4, 98, 50, 0, 65, 1.60], [85, 36.8, 99, 40, 1, 110, 1.75], [75, 37.1, 97, 60, 0, 55, 1.62], [125, 39.0, 88, 80, 1, 75, 1.72], [68, 36.6, 96, 76, 0, 60, 1.60],
    [115, 37.8, 93, 22, 1, 70, 1.80], [70, 36.6, 96, 76, 0, 65, 1.65], [88, 36.5, 95, 77, 1, 80, 1.75], [130, 36.7, 98, 28, 1, 90, 1.85], [65, 38.8, 97, 35, 0, 70, 1.60],
    [75, 36.8, 89, 50, 1, 85, 1.80], [95, 36.6, 96, 40, 0, 120, 1.60], [45, 36.2, 97, 60, 1, 75, 1.75], [82, 36.9, 98, 79, 0, 55, 1.62], [72, 36.5, 96, 45, 1, 50, 1.85],
    [101, 37.4, 96, 20, 1, 70, 1.75], [60, 36.0, 94, 33, 0, 58, 1.68], [74, 36.6, 98, 85, 1, 80, 1.80], [110, 39.5, 91, 5, 1, 18, 1.10], [90, 37.0, 97, 40, 0, 95, 1.60],
    [58, 36.2, 98, 22, 1, 72, 1.82], [120, 37.5, 90, 45, 0, 80, 1.65], [72, 36.6, 98, 30, 1, 70, 1.75], [76, 36.8, 97, 35, 0, 62, 1.68], [140, 37.2, 99, 19, 1, 75, 1.85],
    [66, 35.0, 95, 55, 0, 70, 1.65], [88, 36.9, 85, 65, 1, 90, 1.75], [70, 36.6, 98, 80, 0, 50, 1.55], [102, 38.6, 94, 25, 1, 85, 1.80], [50, 36.4, 96, 42, 0, 60, 1.70],
    [78, 37.4, 97, 77, 1, 88, 1.78], [92, 36.8, 98, 20, 0, 100, 1.55], [70, 36.6, 98, 40, 1, 70, 1.75], [110, 36.9, 97, 30, 0, 55, 1.60], [74, 39.2, 96, 50, 1, 80, 1.75],
    [65, 36.7, 93, 60, 0, 70, 1.65], [105, 37.8, 94, 15, 1, 55, 1.70], [72, 36.6, 98, 28, 0, 55, 1.65], [80, 36.8, 97, 82, 1, 75, 1.70], [118, 38.1, 92, 40, 0, 90, 1.60],
    [60, 36.5, 96, 35, 1, 130, 1.85], [95, 37.5, 95, 25, 0, 65, 1.70], [70, 36.6, 98, 10, 1, 30, 1.40], [130, 39.5, 85, 70, 0, 60, 1.55], [72, 36.6, 98, 35, 1, 78, 1.80],
     [70, 36.5, 98, 22, 1, 70, 1.75], [115, 39.2, 92, 28, 0, 55, 1.60], [60, 36.6, 99, 45, 1, 105, 1.80], [82, 37.1, 94, 65, 0, 75, 1.55], [145, 37.0, 97, 19, 1, 80, 1.85],
    [75, 36.4, 98, 5, 1, 18, 1.10],   [52, 35.8, 96, 70, 0, 60, 1.60],  [90, 38.5, 95, 35, 1, 85, 1.75],  [65, 36.7, 88, 50, 0, 70, 1.65],  [110, 36.8, 96, 82, 1, 65, 1.72],
    [72, 36.6, 98, 25, 0, 52, 1.68],  [102, 37.5, 93, 12, 1, 40, 1.50], [68, 36.5, 97, 77, 0, 58, 1.62], [125, 39.5, 85, 40, 1, 90, 1.80], [88, 36.9, 95, 30, 0, 120, 1.65],
    [62, 36.2, 98, 21, 1, 70, 1.85],  [95, 37.8, 91, 55, 0, 80, 1.58],  [74, 36.8, 97, 85, 1, 75, 1.70], [135, 37.2, 98, 27, 0, 65, 1.70], [58, 36.0, 94, 42, 1, 88, 1.75],
    [80, 36.6, 99, 10, 0, 30, 1.45],  [105, 38.8, 92, 3, 1, 14, 0.95],  [70, 36.7, 96, 68, 0, 72, 1.60], [118, 37.4, 90, 48, 1, 95, 1.78], [66, 36.4, 98, 33, 0, 50, 1.60],
    [78, 36.8, 97, 24, 1, 140, 1.80], [92, 37.6, 95, 15, 0, 45, 1.55],  [50, 36.1, 96, 58, 1, 82, 1.75], [122, 39.1, 87, 72, 0, 63, 1.55], [74, 36.6, 98, 40, 1, 78, 1.82],
    [108, 37.9, 94, 20, 0, 55, 1.65], [64, 36.5, 93, 80, 1, 68, 1.70],  [85, 37.0, 97, 38, 0, 110, 1.60], [130, 38.5, 96, 25, 1, 85, 1.85], [76, 36.7, 98, 29, 0, 60, 1.68],
    [55, 36.3, 91, 62, 1, 85, 1.75],  [98, 37.5, 94, 18, 0, 48, 1.62],  [72, 36.6, 99, 75, 1, 70, 1.75], [112, 39.4, 89, 45, 0, 75, 1.65], [68, 36.8, 97, 10, 1, 35, 1.40],
    [84, 37.2, 95, 55, 0, 90, 1.58],  [140, 37.8, 98, 30, 1, 80, 1.80], [60, 36.5, 92, 79, 0, 55, 1.60], [101, 38.6, 93, 22, 1, 72, 1.78], [74, 36.6, 98, 50, 0, 65, 1.65],
    [95, 37.3, 96, 35, 1, 115, 1.80], [48, 35.9, 97, 65, 0, 60, 1.55],  [120, 36.9, 91, 26, 1, 85, 1.85], [78, 39.0, 95, 12, 0, 42, 1.52], [70, 36.5, 98, 28, 1, 75, 1.78]
]

# Senin 5 kuralına tam uyan, matematiksel olarak doğrulanmış liste:
expected_labels = [
    0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
    1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0,
    1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0
]

print("\n" + "="*145)
print(f"{'NO':<3} | {'BEKLENEN':<8} | {'TAHMİN':<8} | {'GÜVEN':<8} | {'HR':<3} | {'TEMP':<5} | {'SpO2':<4} | {'BMI (OTO)':<9} | {'HRV (EKRAN)':<10} | {'DURUM'}")
print("-" * 145)

correct = 0
for i, s in enumerate(raw_scenarios):
    hr, temp, spo2, age, gen, w, h = s

    # --- HESAPLAMALAR ---
    bmi = w / (h ** 2)
    hrv = 1 / (hr * 0.01) # HRV hesaplanıyor ama modele gönderilmiyor

    # Modele gidecek 8 parametrelik veri paketi (HRV ÇIKARILDI)
    full_data_8 = np.array([hr, temp, spo2, age, gen, w, h, bmi])

    # Manuel Normalizasyon (8 parametre üzerinden)
    scaled_data = (full_data_8 - means) / scales

    # Tahmin
    prob = model.predict(scaled_data.reshape(1, -1), verbose=0)[0][0]

    is_risky = prob >= 0.5
    res_str = "RISKLI" if is_risky else "NORMAL"
    exp_str = "RISKLI" if expected_labels[i] == 1 else "NORMAL"

    status = "✅" if is_risky == (expected_labels[i] == 1) else "❌"
    if status == "✅": correct += 1

    conf = prob * 100 if is_risky else (1 - prob) * 100

    # HRV hala print içinde, sadece kullanıcı görsün diye ekrana basılıyor
    print(f"{i+1:<3} | {exp_str:<8} | {res_str:<8} | %{conf:<6.2f} | {hr:<3} | {temp:<5} | {spo2:<4} | {bmi:<9.2f} | {hrv:<10.2f} | {status}")

print("-" * 145)
print(f"SONUÇ: 100 Senaryoda {correct} Doğru Tahmin! Başarı Oranı: %{(correct/100)*100:.2f}")
print("="* 145)

In [ ]:
# Esp32 için h5 dosyasını model_data.h dosyaına çeviriyor.

import tensorflow as tf
import numpy as np

# 1. Kaydedilen modeli yükle
model = tf.keras.models.load_model('medical_binary_model.h5')

# 2. TFLite Dönüştürücü (Boyutu ESP32 için optimize ediyoruz)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# 3. Byte dizisine çevirme fonksiyonu
def hex_to_c_array(hex_data, var_name):
    c_str = f"const unsigned char {var_name}[] DATA_ALIGN_ATTRIBUTE = {{\n  "
    for i, byte in enumerate(hex_data):
        c_str += f"0x{byte:02x}, "
        if (i + 1) % 12 == 0:
            c_str += "\n  "
    c_str += "\n};\n"
    c_str += f"const int {var_name}_len = {len(hex_data)};"
    return c_str

# 4. model_data.h dosyasını oluştur ve içine yaz
with open('model_data.h', 'w') as f:
    # Header korumaları ve kütüphane eklemeleri
    f.write('#ifndef MODEL_DATA_H\n#define MODEL_DATA_H\n\n')
    f.write('#if defined(__has_attribute) && __has_attribute(aligned)\n')
    f.write('#define DATA_ALIGN_ATTRIBUTE __attribute__((aligned(4)))\n')
    f.write('#else\n')
    f.write('#define DATA_ALIGN_ATTRIBUTE\n')
    f.write('#endif\n\n')

    # Model verisini yaz
    f.write(hex_to_c_array(tflite_model, "medical_model"))
    f.write('\n\n#endif // MODEL_DATA_H')

print("[OK] 'model_data.h' başarıyla oluşturuldu. Dosyayı Arduino projesine ekleyebilirsin.")

In [ ]:
# React'ta yaptığım arayüz örneği için h5 daosyasını uygun formata çeviriyorum.
# Zip den sonra atınca reactta bazı yerleri değiştirmem gerekiyor.

!pip install tensorflowjs

import tensorflow as tf
import tensorflowjs as tfjs
import joblib
from google.colab import files

# 1. Yeni modeli ve scaler'ı yükle
model = tf.keras.models.load_model('medical_binary_model.h5')
scaler = joblib.load('scaler.pkl')  # Bu satır eksikti!

# 2. Kontrol et - (None, 8) çıkmalı
print("Model giriş şekli:", model.input_shape)

# 3. Dönüştür
tfjs.converters.save_keras_model(model, 'tfjs_model_yeni')

# 4. İndir
!zip -r model_yeni.zip tfjs_model_yeni
files.download('model_yeni.zip')

# 5. MEANS ve SCALES - bunları React'a yazacaksın
print("MEANS:", list(scaler.mean_))
print("SCALES:", list(scaler.scale_))

In [ ]:
# Keni verilerimle başka model kaç doğru veriyor. 

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('riskli_temiz_veriseti.csv')

features = ['Heart Rate', 'Body Temperature', 'Oxygen Saturation', 'Age', 
            'Gender', 'Weight (kg)', 'Height (m)', 'Derived_BMI']
X = df[features]
y = df['Risk_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

gnb = GaussianNB()
gnb.fit(X_train, y_train)

raw_scenarios = [
    [72,36.6,98,25,1,75,1.80],[105,36.8,97,30,0,60,1.65],[65,36.5,92,45,1,85,1.75],[80,38.2,96,12,0,40,1.50],[70,36.7,96,78,1,70,1.70],
    [55,36.4,98,50,0,65,1.60],[85,36.8,99,40,1,110,1.75],[75,37.1,97,60,0,55,1.62],[125,39.0,88,80,1,75,1.72],[68,36.6,96,76,0,60,1.60],
    [115,37.8,93,22,1,70,1.80],[70,36.6,96,76,0,65,1.65],[88,36.5,95,77,1,80,1.75],[130,36.7,98,28,1,90,1.85],[65,38.8,97,35,0,70,1.60],
    [75,36.8,89,50,1,85,1.80],[95,36.6,96,40,0,120,1.60],[45,36.2,97,60,1,75,1.75],[82,36.9,98,79,0,55,1.62],[72,36.5,96,45,1,50,1.85],
    [101,37.4,96,20,1,70,1.75],[60,36.0,94,33,0,58,1.68],[74,36.6,98,85,1,80,1.80],[110,39.5,91,5,1,18,1.10],[90,37.0,97,40,0,95,1.60],
    [58,36.2,98,22,1,72,1.82],[120,37.5,90,45,0,80,1.65],[72,36.6,98,30,1,70,1.75],[76,36.8,97,35,0,62,1.68],[140,37.2,99,19,1,75,1.85],
    [66,35.0,95,55,0,70,1.65],[88,36.9,85,65,1,90,1.75],[70,36.6,98,80,0,50,1.55],[102,38.6,94,25,1,85,1.80],[50,36.4,96,42,0,60,1.70],
    [78,37.4,97,77,1,88,1.78],[92,36.8,98,20,0,100,1.55],[70,36.6,98,40,1,70,1.75],[110,36.9,97,30,0,55,1.60],[74,39.2,96,50,1,80,1.75],
    [65,36.7,93,60,0,70,1.65],[105,37.8,94,15,1,55,1.70],[72,36.6,98,28,0,55,1.65],[80,36.8,97,82,1,75,1.70],[118,38.1,92,40,0,90,1.60],
    [60,36.5,96,35,1,130,1.85],[95,37.5,95,25,0,65,1.70],[70,36.6,98,10,1,30,1.40],[130,39.5,85,70,0,60,1.55],[72,36.6,98,35,1,78,1.80],
    [70,36.5,98,22,1,70,1.75],[115,39.2,92,28,0,55,1.60],[60,36.6,99,45,1,105,1.80],[82,37.1,94,65,0,75,1.55],[145,37.0,97,19,1,80,1.85],
    [75,36.4,98,5,1,18,1.10],[52,35.8,96,70,0,60,1.60],[90,38.5,95,35,1,85,1.75],[65,36.7,88,50,0,70,1.65],[110,36.8,96,82,1,65,1.72],
    [72,36.6,98,25,0,52,1.68],[102,37.5,93,12,1,40,1.50],[68,36.5,97,77,0,58,1.62],[125,39.5,85,40,1,90,1.80],[88,36.9,95,30,0,120,1.65],
    [62,36.2,98,21,1,70,1.85],[95,37.8,91,55,0,80,1.58],[74,36.8,97,85,1,75,1.70],[135,37.2,98,27,0,65,1.70],[58,36.0,94,42,1,88,1.75],
    [80,36.6,99,10,0,30,1.45],[105,38.8,92,3,1,14,0.95],[70,36.7,96,68,0,72,1.60],[118,37.4,90,48,1,95,1.78],[66,36.4,98,33,0,50,1.60],
    [78,36.8,97,24,1,140,1.80],[92,37.6,95,15,0,45,1.55],[50,36.1,96,58,1,82,1.75],[122,39.1,87,72,0,63,1.55],[74,36.6,98,40,1,78,1.82],
    [108,37.9,94,20,0,55,1.65],[64,36.5,93,80,1,68,1.70],[85,37.0,97,38,0,110,1.60],[130,38.5,96,25,1,85,1.85],[76,36.7,98,29,0,60,1.68],
    [55,36.3,91,62,1,85,1.75],[98,37.5,94,18,0,48,1.62],[72,36.6,99,75,1,70,1.75],[112,39.4,89,45,0,75,1.65],[68,36.8,97,10,1,35,1.40],
    [84,37.2,95,55,0,90,1.58],[140,37.8,98,30,1,80,1.80],[60,36.5,92,79,0,55,1.60],[101,38.6,93,22,1,72,1.78],[74,36.6,98,50,0,65,1.65],
    [95,37.3,96,35,1,115,1.80],[48,35.9,97,65,0,60,1.55],[120,36.9,91,26,1,85,1.85],[78,39.0,95,12,0,42,1.52],[70,36.5,98,28,1,75,1.78]
]

expected_labels = [
    0,1,1,1,1,1,1,0,1,0,1,0,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,0,0,1,1,1,1,1,1,1,1,0,1,1,
    1,1,1,1,0,1,1,1,1,0,0,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,0,1,1,1,1,1,1,0,1,0,1,1,1,1,0,
    1,1,1,1,0,1,1,0,1,1,1,1,1,1,0,1,1,1,1,0
]

scenarios_with_bmi = []
for s in raw_scenarios:
    hr, temp, spo2, age, gen, w, h = s
    bmi = w / (h ** 2)
    scenarios_with_bmi.append([hr, temp, spo2, age, gen, w, h, bmi])
X_100 = pd.DataFrame(scenarios_with_bmi, columns=features)

# TEST SETİ
rf_test_preds  = rf.predict(X_test)
gnb_test_preds = gnb.predict(X_test)
rf_test_acc    = accuracy_score(y_test, rf_test_preds)  * 100
gnb_test_acc   = accuracy_score(y_test, gnb_test_preds) * 100

# 100 SENARYO
rf_100_preds   = rf.predict(X_100)
gnb_100_preds  = gnb.predict(X_100)
rf_100_acc     = accuracy_score(expected_labels, rf_100_preds)  * 100
gnb_100_acc    = accuracy_score(expected_labels, gnb_100_preds) * 100

# ================================================
# ÖZET TABLO
# ================================================
print("\n" + "="*70)
print("📊 KARŞILAŞTIRMALI MODEL ANALİZİ")
print("="*70)
print(f"{'Model':<35} {'Test Seti':>10} {'100 Senaryo':>12}")
print("-"*70)
print(f"{'Random Forest (GitHub mantığı)':<35} %{rf_test_acc:>8.2f}   %{rf_100_acc:>8.2f}")
print(f"{'Naive Bayes (GitHub ana modeli)':<35} %{gnb_test_acc:>8.2f}   %{gnb_100_acc:>8.2f}")
print(f"{'Benim MLP Modelim':<35} %{'99.49':>8}   %{'96.00':>8}")
print("="*70)

# ================================================
# DETAYLI RAPORLAR
# ================================================
print("\n📋 RANDOM FOREST - Test Seti Detaylı Rapor:")
print(classification_report(y_test, rf_test_preds, target_names=['NORMAL', 'RİSKLİ']))
cm = confusion_matrix(y_test, rf_test_preds)
print(f"  Doğru NORMAL  : {cm[0][0]}  |  Yanlış Alarm : {cm[0][1]}")
print(f"  Kaçan Hasta   : {cm[1][0]}  |  Doğru RİSKLİ: {cm[1][1]}")

print("\n📋 NAİVE BAYES - Test Seti Detaylı Rapor:")
print(classification_report(y_test, gnb_test_preds, target_names=['NORMAL', 'RİSKLİ']))
cm2 = confusion_matrix(y_test, gnb_test_preds)
print(f"  Doğru NORMAL  : {cm2[0][0]}  |  Yanlış Alarm : {cm2[0][1]}")
print(f"  Kaçan Hasta   : {cm2[1][0]}  |  Doğru RİSKLİ: {cm2[1][1]}")

print("\n📋 100 SENARYO - Random Forest Detaylı Rapor:")
print(classification_report(expected_labels, rf_100_preds, target_names=['NORMAL', 'RİSKLİ']))

print("\n📋 100 SENARYO - Naive Bayes Detaylı Rapor:")
print(classification_report(expected_labels, gnb_100_preds, target_names=['NORMAL', 'RİSKLİ']))

print("\n💡 YORUM:")
print("- MLP modeli ESP32'de TFLite ile çalışabilir, RF ve NB çalışamaz.")
print(f"- Naive Bayes test setinde %{gnb_test_acc:.2f} kalırken MLP %99.49 elde etti.")
print("- MLP bağımsız 100 senaryoda %96.00 ile güçlü genelleme yaptı.")